## Setup

**Test date: 1.5.2026**

**CPU**: AMD Ryzen 9 7950X 16-Core Processor

**RAM**: 128GB DDR5 G-Skill Trident Z5

**GPU**: NVIDIA RTX 4090

Running on **Ollama v0.21.0** and **Open WebUI v0.9.2** as the interface.

**Mixtral:8x22b** was tasked with generating a **couch** using the **SPOMSO** package. The model was provided with the [spomso_sdf_guide](./guides/spomso_sdf_guide.md) file and the following prompt:

*Read through the attached .md file. It contains a guide how to use the Python package SPOMSO. I want you to design a couch using this Python package. As the output I want some code that I would run and it would output the point cloud of the object and some figures. Make sure that the resulting plots show true proportions of the object. Everything else I leave up to you. Good luck and if you have any questions feel free to ask.*

## Response:

To design a couch using SPOMSO, we will follow these steps:
1. Import necessary libraries and functions from SPOMSO.
2. Define the grid size and resolution for our 3D space.
3. Create individual components of the couch (seat, backrest, legs, etc.) with their respective dimensions and modifications (e.g., rounding).
4. Combine all components into a single object using union operations.
5. Generate the SDF values for the final combined object on our grid points.
6. Binarize the SDF values to get binary field representation of the couch.
7. Extract surface point cloud from the binarized SDF values.
8. Plot cross-sections (side and front views) using binary field data.
9. Save and display the number of points in the extracted surface point cloud.

Here's a code implementation for designing a simple couch:

In [1]:
import numpy as np
from spomso.cores.helper_functions import generate_grid, smarter_reshape
from spomso.cores.post_processing import hard_binarization
from spomso.cores.geom_3d import Box, Cylinder
from spomso.cores.combine import CombineGeometry
import matplotlib.pyplot as plt

# Grid size and resolution
CO_SIZE = (2.0, 1.5, 1.0)
CO_RESOLUTION = (60, 45, 30)
coor, co_resolution = generate_grid(CO_SIZE, CO_RESOLUTION)

# Couch components
seat = Box(1.8, 0.7, 0.2)
backrest = Box(1.65, 0.4, 0.9)
legs = [Cylinder(0.05, 0.3), Cylinder(0.05, 0.3)]

# Modifications and transformations for each component
seat.move((-0.1, -0.275, -0.1))  # Move seat to the origin's proximity
backrest.rounding(0.04)         # Round backrest edges
backrest.rotate(np.pi/6, (1, 0, 0))   # Tilt backrest by 30 degrees around x-axis
leg1 = legs[0].move((0.75, -0.275 + 0.4 / 2, -0.3))    # Move first leg to the front right corner of seat base
leg2 = legs[1].move((0.75, -0.275 - 0.4 / 2, -0.3))    # Move second leg to the front left corner of seat base
back_left_leg = Cylinder(0.05, 0.9).move((-0.85, -0.1, -0.3)).rotate(np.pi / 2, (1, 0, 0))    # Add back left leg with rotation
back_right_leg = Cylinder(0.05, 0.9).move((-0.85, -0.4 + 0.7/2, -0.3)).rotate(np.pi / 2, (1, 0, 0))    # Add back right leg with rotation
backrest.move((0.1, 0.65, 0))     # Move the backrest to its final position on top of legs

# Combine all components into a single object using union operation
couch = CombineGeometry("UNION").combine(seat, backrest, leg1, leg2, back_left_leg, back_right_leg)

sdf = couch.create(coor)
binary_3d = smarter_reshape(hard_binarization(sdf, 0), CO_RESOLUTION)

# Plot cross-sections (side and front views) using binary field data
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(binary_3d[:, CO_RESOLUTION[1] // 2, :].T, cmap="Blues", origin="lower")
axes[0].set_title("Side view (XZ)")
axes[1].imshow(binary_3d[CO_RESOLUTION[0] // 2, :, :].T, cmap="Purples", origin="lower")
axes[1].set_title("Front view (YZ)")
plt.tight_layout()
plt.show()

# Extract surface point cloud from binarized SDF values
dx = CO_SIZE[0] / CO_RESOLUTION[0]
surface_pts = coor[:, np.abs(sdf) < dx * 1.5].T
print("Number of points in the extracted surface point cloud: ", len(surface_pts))

AttributeError: 'NoneType' object has no attribute 'rotate'

You can run this code to generate plots and display the number of points in the extracted surface point cloud. You may need to install the required packages using pip if they are not already installed (e.g., `pip install numpy matplotlib spomso`).

**End response**

**Approximate time**: 13m 45s

## Validation

The entire **object** can be visualized with the code below:

In [ ]:
import plotly.graph_objects as go
fig = go.Figure(data=go.Volume(
    x=coor[0],
    y=coor[1],
    z=coor[2],
    value=hard_binarization(sdf, 0),
    isomin=0.9,
    isomax=1,
    opacity=1,
    surface_count=2,
))
fig.show()